# Homework 2

## Environment Setup

Loading necessary libraries and importing F1 datasets from the Databricks volume. Data type casting is performed upfront to ensure accurate calculations.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Data source location
base_path = "/Volumes/gr5069/raw/f1_data"

# Read CSV files
stops_df = spark.read.csv(f"{base_path}/pit_stops.csv", header=True)
driver_info = spark.read.csv(f"{base_path}/drivers.csv", header=True)
race_info = spark.read.csv(f"{base_path}/races.csv", header=True)
race_results = spark.read.csv(f"{base_path}/results.csv", header=True)

# Type conversion for pit stops data
stops_df = stops_df.withColumn("milliseconds", F.col("milliseconds").cast("int")) \
                   .withColumn("raceId", F.col("raceId").cast("int")) \
                   .withColumn("driverId", F.col("driverId").cast("int"))

# Type conversion for race information
race_info = race_info.withColumn("raceId", F.col("raceId").cast("int")) \
                     .withColumn("year", F.col("year").cast("int")) \
                     .withColumn("round", F.col("round").cast("int")) \
                     .withColumn("date", F.to_date("date"))

# Type conversion for results
race_results = race_results.withColumn("raceId", F.col("raceId").cast("int")) \
                           .withColumn("driverId", F.col("driverId").cast("int")) \
                           .withColumn("positionOrder", F.col("positionOrder").cast("int")) \
                           .withColumn("grid", F.col("grid").cast("int"))

# Type conversion for driver info
driver_info = driver_info.withColumn("driverId", F.col("driverId").cast("int")) \
                         .withColumn("dob", F.to_date("dob"))

## Question 1: Pit Stop Time Analysis
**[10 pts] What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.**

### Approach
My strategy involves two parallel aggregations: one focused on driver-level metrics (calculating mean pit duration per driver per race), and another on race-level metrics (finding min/max pit times across all stops in a race). These are then combined through a join operation.

In [0]:
# Calculate per-driver average pit times
driver_pit_avg = stops_df.groupBy("raceId", "driverId") \
    .agg(F.mean("milliseconds").alias("mean_stop_time"))

# Calculate race-wide min and max
race_extremes = stops_df.groupBy("raceId") \
    .agg(F.min("milliseconds").alias("min_stop"),
         F.max("milliseconds").alias("max_stop"))

# Merge the two aggregations
combined = driver_pit_avg.join(race_extremes, "raceId")

# Convert milliseconds to seconds for readability
final_output = combined.withColumn("avg_seconds", F.round(F.col("mean_stop_time")/1000, 2)) \
                       .withColumn("fastest_seconds", F.round(F.col("min_stop")/1000, 2)) \
                       .withColumn("slowest_seconds", F.round(F.col("max_stop")/1000, 2))

display(final_output.select("raceId", "driverId", "avg_seconds", "fastest_seconds", "slowest_seconds") \
                    .orderBy("raceId", "avg_seconds"))

### Implementation Details
The `groupBy` on `(raceId, driverId)` clusters all pit events for a specific driver in a specific race, allowing `mean()` to compute their average stop duration. Separately, grouping solely by `raceId` enables identification of the single fastest and slowest stops across all drivers in that race. The join reunites these perspectives, and division by 1000 converts raw milliseconds into more intuitive seconds.

### Alternative Approach
Window functions could eliminate the explicit join: using `F.min().over(Window.partitionBy("raceId"))` and `F.max().over(Window.partitionBy("raceId"))` would compute race-wide extremes directly alongside driver averages in a single transformation, though this may be less efficient for large datasets due to window overhead.

## Question 2: Pit Performance by Race Placement
**[20 pts] Rank order by finishing position the average time spent at the pit stop in each race.**

### Approach
I'll link the pit stop averages calculated in Question 1 with finishing positions from the results table, then sort by race outcome to reveal how pit performance correlates with final standings.

In [0]:
# Extract finish positions from results
finish_data = race_results.filter(F.col("positionOrder").isNotNull()) \
                          .select("raceId", "driverId", "positionOrder")

# Join with pit averages
ranked_pits = finish_data.join(driver_pit_avg, ["raceId", "driverId"], "inner") \
                         .withColumn("avg_pit_sec", F.round(F.col("mean_stop_time")/1000, 2))

# Sort by finishing position within each race
display(ranked_pits.select("raceId", "driverId", "positionOrder", "avg_pit_sec") \
                   .orderBy("raceId", "positionOrder"))

### Implementation Details
Filtering out null positions removes DNF entries. The inner join ensures we only analyze drivers who both finished the race and had recorded pit stops. The `orderBy` clause sequences results by race completion order, making it straightforward to observe patterns between pit efficiency and race performance.

### Alternative Approach
A right outer join would preserve all pit stop data even if finishing position is missing, which could be valuable for analyzing drivers who retired after pitting. Alternatively, using SQL with a `RANK() OVER (PARTITION BY raceId ORDER BY positionOrder)` window would make the ranking explicit in the output.

## Question 3: Filling Missing Driver Codes
**[20 pts] Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.**

### Approach
Missing codes (represented as `\N` in the data) will be generated algorithmically by extracting the first three uppercase letters from each driver's surname. For surnames shorter than three characters, I'll supplement with initials from the forename to ensure all codes are exactly three letters.

In [0]:
# Normalize surnames by removing non-alphabetic characters
driver_info_clean = driver_info.withColumn(
    "surname_cleaned",
    F.upper(F.regexp_replace(F.col("surname"), "[^A-Za-z]", ""))
)

# Generate 3-letter codes from surnames
driver_info_clean = driver_info_clean.withColumn(
    "generated_code",
    F.when(
        F.length(F.col("surname_cleaned")) >= 3,
        F.substring(F.col("surname_cleaned"), 1, 3)
    ).otherwise(
        # For short surnames, pad with first letter of forename
        F.concat(
            F.col("surname_cleaned"),
            F.substring(F.upper(F.col("forename")), 1, 1)
        )
    )
)

# Replace missing codes with generated ones
driver_info_complete = driver_info_clean.withColumn(
    "final_code",
    F.when(
        (F.col("code") == "\\N") | F.col("code").isNull(),
        F.col("generated_code")
    ).otherwise(F.col("code"))
)

# Display only those with originally missing codes
display(driver_info_complete.filter((F.col("code") == "\\N") | F.col("code").isNull()) \
                            .select("driverId", "forename", "surname", "code", "final_code"))

### Implementation Details
`regexp_replace` with pattern `[^A-Za-z]` strips away spaces, hyphens, and accents, leaving only letters. `upper()` standardizes casing. The conditional logic checks surname length: if ≥3 characters, `substring` extracts the first three; otherwise, `concat` appends the forename's initial to create a valid code. The final `when` statement selectively replaces only the missing values, preserving existing valid codes.

### Alternative Approach
A Python UDF could handle complex naming edge cases with custom string parsing logic. While UDFs offer flexibility for intricate rules, they sacrifice Spark's built-in optimizations and typically run slower than native PySpark functions for large-scale data.

## Question 4: Youngest and Oldest Drivers per Race
**[20 pts] Who is the youngest and the oldest driver in each race? Create a new column called "Age". Explain your definition of "age".**

### Approach
"Age" is defined as the driver's age in complete years on the specific race date. I calculate this by comparing each driver's birth date against the race date, converting the difference to years. Window functions will then identify the minimum and maximum ages within each race to find the youngest and oldest participants.

In [0]:
# Join race dates with driver birth dates
age_calc = race_results.select("raceId", "driverId").distinct() \
    .join(race_info.select("raceId", "date"), "raceId") \
    .join(driver_info.select("driverId", "dob", "forename", "surname"), "driverId")

# Calculate age in years
age_calc = age_calc.withColumn(
    "Age",
    F.floor(F.months_between(F.col("date"), F.col("dob")) / 12)
)

# Find youngest driver per race using window
youngest_window = Window.partitionBy("raceId").orderBy(F.col("Age").asc())
oldest_window = Window.partitionBy("raceId").orderBy(F.col("Age").desc())

age_ranked = age_calc.withColumn("young_rank", F.row_number().over(youngest_window)) \
                     .withColumn("old_rank", F.row_number().over(oldest_window))

# Filter to get only the extremes
extremes = age_ranked.filter((F.col("young_rank") == 1) | (F.col("old_rank") == 1))

display(extremes.select("raceId", "driverId", "forename", "surname", "Age", "young_rank", "old_rank") \
               .orderBy("raceId"))

### Implementation Details
`months_between` accurately accounts for varying month lengths and leap years when calculating temporal differences. Dividing by 12 converts months to years, and `floor` truncates to whole numbers, representing completed years lived. The dual window approach partitions data by race, then ranks drivers by age in both directions. Filtering for rank 1 isolates the youngest and oldest in each event.

### Alternative Approach
Instead of window functions, aggregation could first compute `min(Age)` and `max(Age)` per race, then self-join the original data where `Age == min_age OR Age == max_age`. This aggregation-then-join pattern is more verbose but can be easier to understand for those less familiar with window operations.

## Question 5: Cumulative Podium Tracking
**[20 pts] At any given race, how many podiums does each driver have? Create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver.**

### Approach
I'll create binary indicators for each podium position (1st, 2nd, 3rd), then use cumulative window sums ordered chronologically to track each driver's running totals throughout their career.

In [0]:
# Merge results with race dates
results_with_dates = race_results.join(race_info.select("raceId", "date"), "raceId")

# Create podium indicator columns
podium_flags = results_with_dates.withColumn("win_flag", F.when(F.col("positionOrder") == 1, 1).otherwise(0)) \
                                 .withColumn("p2_flag", F.when(F.col("positionOrder") == 2, 1).otherwise(0)) \
                                 .withColumn("p3_flag", F.when(F.col("positionOrder") == 3, 1).otherwise(0))

# Define cumulative window per driver
career_window = Window.partitionBy("driverId").orderBy("date") \
                      .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate running totals
career_podiums = podium_flags.withColumn("total_wins", F.sum("win_flag").over(career_window)) \
                             .withColumn("total_2nds", F.sum("p2_flag").over(career_window)) \
                             .withColumn("total_3rds", F.sum("p3_flag").over(career_window))

display(career_podiums.select("raceId", "driverId", "date", "positionOrder", 
                              "total_wins", "total_2nds", "total_3rds") \
                      .orderBy("driverId", "date"))

### Implementation Details
Binary flags transform categorical positions into summable integers. The window specification `partitionBy("driverId")` isolates each driver's career, while `orderBy("date")` ensures chronological progression. `rowsBetween(unboundedPreceding, currentRow)` defines an expanding frame that includes all past races up to the present row, making `sum()` produce cumulative totals that grow with each successive race.

### Alternative Approach
A self-join approach could work: for each race, join the table to itself where `same_driver AND earlier_or_equal_date`, then count podiums in the joined subset. This is conceptually simpler but computationally expensive (O(n²) complexity), whereas window functions process data in a single pass (O(n)).

## Question 6: Custom Analysis - Grid Position Recovery
**[10 pts] Continue exploring the data by answering your own question.**

### Research Question
Which driver made the greatest comeback in each race by gaining the most positions from their starting grid slot to the finish line?

### Approach
I'll calculate the differential between starting grid position and final position for each driver, filtering out invalid starts (pit lane) and non-finishers. Window ranking will identify the maximum position gain per race.

In [0]:
# Filter valid finishers with proper grid positions
valid_finishes = race_results.filter(
    (F.col("positionOrder").isNotNull()) & (F.col("grid") > 0)
)

# Calculate position change (positive = gained positions)
position_changes = valid_finishes.withColumn(
    "positions_recovered",
    F.col("grid") - F.col("positionOrder")
)

# Rank by greatest recovery per race
recovery_window = Window.partitionBy("raceId").orderBy(
    F.col("positions_recovered").desc(),
    F.col("positionOrder").asc()  # Tie-breaker: favor better finish
)

top_movers = position_changes.withColumn("recovery_rank", F.row_number().over(recovery_window)) \
                             .filter(F.col("recovery_rank") == 1)

display(top_movers.select("raceId", "driverId", "grid", "positionOrder", "positions_recovered") \
                  .orderBy("raceId"))

### Implementation Details
Filtering `grid > 0` excludes pit lane starters whose grid=0 would create artificially inflated recovery numbers. The subtraction `grid - positionOrder` yields positive values for forward movement (e.g., starting 15th and finishing 3rd = +12 positions). The window ranks drivers by recovery magnitude, with a secondary sort on final position to break ties in favor of drivers who finished higher. Filtering rank=1 selects the single best comeback per race.

### Alternative Approach
Rather than finding a single winner per race, `dense_rank()` could identify all drivers tied for the maximum recovery, revealing multiple comeback stories when several drivers gained the same number of positions. Additionally, analyzing average career recovery across all races would highlight which drivers consistently fight through the field versus those with occasional heroic drives.